In [ ]:
# Clone repo (for Google Colab – safe to re-run; no duplicate clone)
import os
REPO_DIR = "/content/Sigmoid-TopK-Fusion"  # Colab starts in /content
os.chdir("/content")
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/sriharshapy/Sigmoid-TopK-Fusion.git
%cd /content/Sigmoid-TopK-Fusion

# Sigmoid + Top-K MoE: Benchmark & NCU Profiling

Runs both the **PyTorch** and **Triton** sigmoid+top-k implementations, then profiles them with **NVIDIA Nsight Compute (ncu)** and merges metrics into a CSV.

Pattern follows the [High-Performance-Reduction-Kernels](https://github.com/sriharshapy/High-Performance-Reduction-Kernels/tree/main/reductions/sum) example: warmup, timed runs, then NCU with `--target-processes application-only`, metrics export to CSV, and merge.

## 1. Setup: ensure tensor.pt exists

In [ ]:
import os
import subprocess
from pathlib import Path

ROOT = Path.cwd()
os.chdir(ROOT)
TENSOR_FILE = "tensor.pt"

if not Path(TENSOR_FILE).exists():
    print(f"Creating {TENSOR_FILE} with default 4x6 tensor...")
    subprocess.run(["python", "save_tensor.py", "-n", "16384", "-m", "128", "-o", TENSOR_FILE], check=True)
else:
    print(f"Using existing {TENSOR_FILE}")
print("Working directory:", os.getcwd())

## 2. Run both scripts (benchmark: 3 warmup, 100 iters)

In [ ]:
!python sigmoid_topk_moe_fused_pytorch.py -f tensor.pt -k 2 -n 100

In [ ]:
!python sigmoid_topk_moe_fused_triton.py -f tensor.pt -k 2 -n 100

## 3. NCU profiling (detailed GPU metrics)

Runs `ncu` on both implementations with `--no-warmup` and 1 iteration, then exports raw CSV and merges. **Requires:** NVIDIA GPU, Nsight Compute installed, and scripts run on GPU (tensor on CUDA). If your tensor is on CPU, move it to CUDA first (e.g. in `save_tensor.py` or load with `map_location='cuda'`) for meaningful NCU results.

In [ ]:
import csv
import io
import subprocess
import sys
from pathlib import Path

TENSOR_FILE = "tensor.pt"
OUTPUT_CSV = "ncu_merged.csv"
NCU = "ncu"
PYTHON = "python"
TRITON_SCRIPT = "sigmoid_topk_moe_fused_triton.py"
TORCH_SCRIPT = "sigmoid_topk_moe_fused_pytorch.py"

# One iteration, no warmup for clean NCU trace
APP_ARGS = ["-f", TENSOR_FILE, "-k", "2", "-n", "1", "--no-warmup"]

NCU_METRICS = (
    "gpu__time_duration.sum,"
    "dram__throughput.avg.pct_of_peak_sustained_elapsed,"
    "sm__warps_active.avg.pct_of_peak_sustained_active,"
    "smsp__thread_inst_executed_per_inst_executed.ratio,"
    "l1tex__throughput.avg.pct_of_peak_sustained_elapsed,"
    "l1tex__data_bank_conflicts_pipe_lsu_mem_shared_op_ld.sum,"
    "lts__t_sectors.avg.pct_of_peak_sustained_elapsed,"
    "l1tex__t_sector_hit_rate.pct,"
    "lts__t_sectors_srcunit_tex_op_read.sum,"
    "smsp__sass_average_data_bytes_per_sector_mem_global_op_ld.pct,"
    "dram__bytes_read.sum,"
    "dram__bytes_write.sum",
)

def run(cmd, timeout_s=1800):
    return subprocess.run(cmd, capture_output=True, text=True, timeout=timeout_s)

def profile_one(impl, run_cmd, kernel_regex=None, launch_count=None):
    rep = Path(f"report_{impl}.ncu-rep")
    cmd = [
        NCU,
        "--target-processes", "application-only",
        "--metrics", NCU_METRICS,
        "-o", str(rep),
        "-f",
    ]
    if kernel_regex is not None:
        cmd += ["-k", f"regex:{kernel_regex}"]
    if launch_count is not None:
        cmd += ["-c", str(launch_count)]
    cmd += ["--"] + run_cmd

    print(f"[{impl}] {" ".join(cmd)}")
    r = run(cmd)
    if r.returncode != 0:
        print(f"[{impl}] ncu failed (returncode={r.returncode})", file=sys.stderr)
        if r.stdout:
            print(r.stdout[:2000], file=sys.stderr)
        if r.stderr:
            print(r.stderr[:2000], file=sys.stderr)
        return None
    if not rep.exists():
        print(f"[{impl}] Report missing: {rep}", file=sys.stderr)
        return None
    return rep

def export_csv(rep):
    cmd = [NCU, "-i", str(rep), "--csv", "--page", "raw"]
    r = run(cmd, timeout_s=180)
    if r.returncode != 0:
        print(f"[export] failed for {rep}", file=sys.stderr)
        return None
    return r.stdout

def parse_raw_csv(csv_text, impl):
    rows = list(csv.reader(io.StringIO(csv_text)))
    if not rows:
        return None
    header = rows[0]
    data_start = 1
    if len(rows) > 1 and rows[1] and rows[1][0] == "" and (
        "byte" in str(rows[1]).lower() or "ns" in str(rows[1]).lower() or "%" in str(rows[1])
    ):
        data_start = 2
    if "implementation" not in header:
        header = ["implementation"] + header
    n_cols = len(header) - 1
    out_rows = []
    for r in rows[data_start:]:
        if not r or (len(r) == 1 and not r[0].strip()):
            continue
        r = (r + [""] * n_cols)[:n_cols]
        out_rows.append([impl] + r)
    return header, out_rows

# Profile Triton (match our JIT kernel name)
reports = []
reports.append(("triton", profile_one(
    "triton",
    [PYTHON, TRITON_SCRIPT] + APP_ARGS,
    kernel_regex=r"sigmoid_topk",
    launch_count=1,
)))

# Profile PyTorch (capture relevant kernels; no filter = all kernels)
reports.append(("pytorch", profile_one(
    "pytorch",
    [PYTHON, TORCH_SCRIPT] + APP_ARGS,
    launch_count=None,
)))

reports = [(impl, rep) for impl, rep in reports if rep is not None]
if not reports:
    raise RuntimeError("No .ncu-rep reports produced. Ensure GPU + ncu on PATH.")

merged_header = None
merged_rows = []
for impl, rep in reports:
    txt = export_csv(rep)
    if not txt:
        continue
    parsed = parse_raw_csv(txt, impl)
    if not parsed:
        continue
    header, rows = parsed
    if merged_header is None:
        merged_header = header
    merged_rows.extend(rows)

if not merged_header or not merged_rows:
    raise RuntimeError("No CSV rows produced")

out_path = Path(OUTPUT_CSV)
with open(out_path, "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(merged_header)
    w.writerows(merged_rows)

print(f"Done. Wrote merged CSV: {out_path.resolve()}")
print(f"Rows: {len(merged_rows)} (one per kernel launch)")

## 4. View merged NCU metrics

In [ ]:
import pandas as pd
from pathlib import Path

csv_path = Path("ncu_merged.csv")
if not csv_path.is_file():
    raise FileNotFoundError(f"Run the NCU cell first. Not found: {csv_path}")

df = pd.read_csv(csv_path)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
df